# Week 7 V2 Trial-8 Audit

Restore the preserved accepted trial-8 adapter from its GitHub Release asset, run the complete held-out evaluation, and save a separate audit report without changing the original Week 7 results.

## 1. Runtime

Select a GPU runtime. Add `GITHUB_TOKEN` in Colab Secrets and grant this notebook access.

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes pandas numpy safetensors

## 2. GitHub workspace

In [ ]:
from collections import deque
from pathlib import Path
import subprocess
import sys

REPOSITORY = "HannanSeyfi/unlearning-thesis"
BRANCH = "main"
FALLBACK_BRANCH = "codex/week7-rollback-constrained-v2"
REPO_DIR = Path("/content/unlearning-thesis")
SPARSE_PATHS = [
    "Tools",
    "Week 3.5/data/synthetic_facts_v1",
    "Week 3.5/data/general_controls_v1",
    "Week 6/constrained_gradient_unlearning",
    "Week 7/adaptive_constrained_unlearning",
    "Week 7/rollback_constrained_unlearning_v2",
    "Week 7/results/rollback_constrained_unlearning_v2/results",
    "Week 7/results/rollback_constrained_unlearning_v2/resume_state/candidates/r02_rollback_lab_guarded",
    "Week 7/results/rollback_constrained_unlearning_v2_trial8_audit",
]

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, f"https://github.com/{REPOSITORY}.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "sparse-checkout", "set", "--cone", *SPARSE_PATHS], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)

%cd {REPO_DIR}
from Tools.github_colab_sync import setup_colab_repo

THESIS_DIR = setup_colab_repo(REPOSITORY, BRANCH, REPO_DIR)
%cd {THESIS_DIR}

SCRIPT_PATH = THESIS_DIR / "Week 7" / "rollback_constrained_unlearning_v2" / "evaluate_week7_v2_trial8_adapter.py"
if not SCRIPT_PATH.exists():
    print(f"Audit script missing on {BRANCH}; checking out {FALLBACK_BRANCH}.")
    subprocess.run(["git", "fetch", "origin", FALLBACK_BRANCH], cwd=THESIS_DIR, check=True)
    subprocess.run(["git", "checkout", "-B", FALLBACK_BRANCH, "FETCH_HEAD"], cwd=THESIS_DIR, check=True)
    BRANCH = FALLBACK_BRANCH

print("Using branch:", BRANCH)
print("Audit script:", SCRIPT_PATH)

## 3. Verify the preserved checkpoint

In [ ]:
import json

STATE_PATH = THESIS_DIR / "Week 7" / "results" / "rollback_constrained_unlearning_v2" / "resume_state" / "candidates" / "r02_rollback_lab_guarded" / "state.json"
state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
assert int(state["epoch"]) == 8, state
print(json.dumps({key: state.get(key) for key in ["candidate_id", "epoch", "accepted_blocks", "release_tag", "release_asset"]}, indent=2))

## 4. Restore and evaluate trial 8

In [ ]:
AUDIT_RUN_NAME = "rollback_constrained_unlearning_v2_trial8_audit"
RESET_AUDIT_RESULTS = True

cmd = [
    sys.executable,
    "-u",
    str(SCRIPT_PATH),
    "--repo-root", str(THESIS_DIR),
    "--audit-run-name", AUDIT_RUN_NAME,
    "--push-results",
    "--push-branch", BRANCH,
]
if RESET_AUDIT_RESULTS:
    cmd.append("--reset")

print("Running:", " ".join(cmd), flush=True)
process = subprocess.Popen(cmd, cwd=THESIS_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
output_tail = deque(maxlen=120)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
    output_tail.append(line)
return_code = process.wait()
if return_code != 0:
    failure_path = THESIS_DIR / "Week 7" / "results" / AUDIT_RUN_NAME / "failure.json"
    if failure_path.exists():
        print(failure_path.read_text(encoding="utf-8"), flush=True)
    raise RuntimeError(f"Trial-8 audit exited with status {return_code}. Last output:\n{''.join(output_tail)}")

## 5. Review the audit

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

AUDIT_DIR = THESIS_DIR / "Week 7" / "results" / AUDIT_RUN_NAME / "results"
display(pd.read_csv(AUDIT_DIR / "trial8_cross_week_comparison.csv"))
display(Markdown((AUDIT_DIR / "trial8_audit_report.md").read_text(encoding="utf-8")))
print("Audit outputs pushed from:", AUDIT_DIR.parent)